# latent analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

PHASE_ENCODER_PATH = "/path/to/public/AE_encoder_phase"  

ROWS, COLS = 4, 5
N = ROWS * COLS
tile_h = 14
pad_v, pad_h = 10, 12

M_FIX = 1.4
CHI_FIX = 0
LAMBDA_FIX = 0

def build_canvas(rows_01, D, left_labels, savepath):
    lm = 110
    H_total = ROWS * tile_h + (ROWS - 1) * pad_v
    W_total = lm + COLS * D + (COLS - 1) * pad_h
    canvas = np.zeros((H_total, W_total), dtype=float)

    def place_tile(r, c, vec01):
        by = r * (tile_h + pad_v)
        bx = lm + c * (D + pad_h)
        tile = np.repeat(vec01.reshape(1, -1), tile_h, axis=0)
        canvas[by:by+tile_h, bx:bx+D] = tile

    fig = plt.figure(figsize=(10, 2.0))
    ax = plt.gca()

    k = 0
    for r in range(ROWS):
        for c in range(COLS):
            place_tile(r, c, rows_01[k]); k += 1

    ax.imshow(canvas, aspect="auto", interpolation="nearest", zorder=1, cmap="viridis")
    plt.xticks([]); plt.yticks([])

    for r in range(ROWS):
        y_center = r * (tile_h + pad_v) + tile_h / 2.0
        plt.text(6, y_center, left_labels[r], color="black", fontsize=10, va="center", ha="left", zorder=3)

    for k50 in range(50, D, 50):
        for c in range(COLS):
            xk = lm + c*(D+pad_h) + k50
            plt.vlines(xk, 0, H_total, linewidth=0.2, colors="k", alpha=0.25)

    plt.tight_layout()
    plt.savefig(savepath, dpi=220, bbox_inches="tight")
    plt.close(fig)

def sort_by_sensitivity(rows_delta):
    sens = np.sqrt(np.sum(rows_delta**2, axis=0))
    order = np.argsort(-sens)
    return rows_delta[:, order]

def to_rows01_abs_q(rows_delta_sorted, q=99.0, global_scale=None):
    A = np.abs(rows_delta_sorted)
    scale = global_scale if global_scale is not None else np.percentile(A, q)
    rowsn = np.clip(A / (scale + 1e-12), 0.0, 1.0)
    return rowsn


def load_phase_encoder():
    return tf.keras.models.load_model(PHASE_ENCODER_PATH)

def encode_phase(model, X, batch_size=512):
    Z = model.predict(X, batch_size=batch_size, verbose=0)
    return Z

def build_batch(m1, m2, L1, L2, s1x, s1y, s1z, s2x, s2y, s2z):
    return np.stack([m1, m2, L1, L2, s1x, s1y, s1z, s2x, s2y, s2z], axis=1).astype(np.float32)

def case_tidal():
    lam = np.linspace(0, 1000, N).astype(np.float32)
    m1 = np.full(N, M_FIX, dtype=np.float32)
    m2 = np.full(N, M_FIX, dtype=np.float32)
    L1 = lam; L2 = lam
    s1x = np.zeros(N, np.float32); s1y = np.zeros(N, np.float32); s1z = np.full(N, CHI_FIX, np.float32)
    s2x = np.zeros(N, np.float32); s2y = np.zeros(N, np.float32); s2z = np.full(N, CHI_FIX, np.float32)
    X = build_batch(m1, m2, L1, L2, s1x, s1y, s1z, s2x, s2y, s2z)

    left_labels = []
    for r in range(ROWS):
        v0 = int(lam[r*COLS + 0]); v1 = int(lam[r*COLS + (COLS-1)])
        left_labels.append(r"$\Lambda_1{=}\Lambda_2$ " + f"{v0}→{v1}")
    return X, left_labels, "latent_lambda.png"

def case_mass():
    Mtot = np.linspace(2.00, 2.84, N).astype(np.float32)
    m1 = Mtot/2.0; m2 = Mtot/2.0
    L1 = np.full(N, LAMBDA_FIX, np.float32); L2 = np.full(N, LAMBDA_FIX, np.float32)
    s1x = np.zeros(N, np.float32); s1y = np.zeros(N, np.float32); s1z = np.full(N, CHI_FIX, np.float32)
    s2x = np.zeros(N, np.float32); s2y = np.zeros(N, np.float32); s2z = np.full(N, CHI_FIX, np.float32)
    X = build_batch(m1, m2, L1, L2, s1x, s1y, s1z, s2x, s2y, s2z)

    left_labels = []
    for r in range(ROWS):
        v0 = Mtot[r*COLS + 0]; v1 = Mtot[r*COLS + (COLS-1)]
        left_labels.append(f"$M_{{\\rm tot}}$ {v0:.2f}→{v1:.2f}")
    return X, left_labels, "latent_mass.png"

def case_spin():
    chi = np.linspace(0.0, 0.5, N).astype(np.float32)
    m1 = np.full(N, M_FIX, np.float32); m2 = np.full(N, M_FIX, np.float32)
    L1 = np.full(N, LAMBDA_FIX, np.float32); L2 = np.full(N, LAMBDA_FIX, np.float32)
    s1x = np.zeros(N, np.float32); s1y = np.zeros(N, np.float32); s1z = chi
    s2x = np.zeros(N, np.float32); s2y = np.zeros(N, np.float32); s2z = chi
    X = build_batch(m1, m2, L1, L2, s1x, s1y, s1z, s2x, s2y, s2z)

    left_labels = []
    for r in range(ROWS):
        v0 = chi[r*COLS + 0]; v1 = chi[r*COLS + (COLS-1)]
        left_labels.append(r"$\chi_{\rm tot}$ " + f"{v0:.2f}→{v1:.2f}")
    return X, left_labels, "latent_spin.png"


enc = load_phase_encoder()

X_t, lab_t, out_t = case_tidal()
X_m, lab_m, out_m = case_mass()
X_s, lab_s, out_s = case_spin()

Z_t = encode_phase(enc, X_t)
Z_m = encode_phase(enc, X_m)
Z_s = encode_phase(enc, X_s)

rows_t = sort_by_sensitivity(Z_t - Z_t[0:1, :])
rows_m = sort_by_sensitivity(Z_m - Z_m[0:1, :])
rows_s = sort_by_sensitivity(Z_s - Z_s[0:1, :])

glob_scale = np.percentile(np.abs(np.concatenate([rows_t.ravel(), rows_m.ravel(), rows_s.ravel()])), 99.0)

A = to_rows01_abs_q(rows_t, q=99.0, global_scale=glob_scale)
B = to_rows01_abs_q(rows_m, q=99.0, global_scale=glob_scale)
C = to_rows01_abs_q(rows_s, q=99.0, global_scale=glob_scale)

D = A.shape[1]
build_canvas(A, D, lab_t, out_t)
build_canvas(B, D, lab_m, out_m)
build_canvas(C, D, lab_s, out_s)




# phi_time

In [ ]:

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from pycbc import waveform
import pycbc.waveform.utils  
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.patches import Rectangle, ConnectionPatch
from matplotlib.ticker import FormatStrFormatter




OUTPUT_DIR      = "./figs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
sample_frequency= 2048                
f_lower         = 50                   
T_total         = 2.0                 
window_sec      = 0.1                  
unwrap_phase    = True                 

m1, m2          = 1.40, 1.40
spin1 = dict(x=0.2, y=0.2, z=0.2)
spin2 = dict(x=0.2, y=0.2, z=0.2)

lambda_vals     = np.linspace(0, 500, 6, dtype=int)
lambda_pairs    = [(int(l), int(l)) for l in lambda_vals]   


def cal_pad_array(arr,target_length):
    if len(arr) < target_length:
        result = np.pad(arr, (0, target_length - len(arr)), 'constant', constant_values=0)
    else:
        result = arr[-target_length:]
    return result

def cal_amplitude_from_hphc_pycbc(hp, hc):
    amplitude = waveform.utils.amplitude_from_polarizations(hp, hc)
    return amplitude

def cal_phase_from_hphc_pycbc(hp, hc):
    phase = waveform.utils.phase_from_polarizations(hp, hc)
    phase=phase-phase[0]
    return phase

def _phase_amp_from_hphc(hp, hc):
    hp = hp.trim_zeros(); hc = hc.trim_zeros()
    amplitude = cal_amplitude_from_hphc_pycbc(hp, hc)
    phase     = cal_phase_from_hphc_pycbc(hp, hc)
    amplitude = np.asarray(amplitude, dtype=np.float32)
    phase     = np.asarray(phase,     dtype=np.float32)
    return amplitude, phase

def _align_to_merger(amplitude, phase, fs, T_total, do_unwrap=True):

    target_len = int(T_total * fs)
    amplitude  = cal_pad_array(amplitude, target_len)
    phase      = cal_pad_array(phase,     target_len)

    if do_unwrap:
        phase = np.unwrap(phase)
    phase = phase - phase[0]

    peak_idx = int(np.argmax(amplitude))
    n = len(phase)
    t = np.arange(n, dtype=np.float32) / np.float32(fs)
    t_rel = t - t[peak_idx] 

    return t_rel, phase, amplitude, peak_idx

def cal_generate_BNS_waveform(approximant='IMRPhenomXP_NRTidalv2', **kwargs):
    waveform_args = {"approximant": approximant}
    for key, value in kwargs.items():
        if value is not None: 
            waveform_args[key] = value
    hp, hc = waveform.get_td_waveform(**waveform_args)
    return hp, hc


def generate_phase_curve(m1, m2, spin1, spin2, L1, L2, fs, f_low, T):
    hp, hc = cal_generate_BNS_waveform(
        mass1=m1, mass2=m2,
        delta_t=1.0/fs, f_lower=f_low,
        spin1x=spin1["x"], spin1y=spin1["y"], spin1z=spin1["z"],
        spin2x=spin2["x"], spin2y=spin2["y"], spin2z=spin2["z"],
        lambda1=L1, lambda2=L2,
    )
    amplitude, phase = _phase_amp_from_hphc(hp, hc)
    return _align_to_merger(amplitude, phase, fs, T, do_unwrap=unwrap_phase)

curves = [] 
for (L1, L2) in tqdm(lambda_pairs, desc="Generating curves"):
    t_rel, phi, amp, peak_idx = generate_phase_curve(m1, m2, spin1, spin2, L1, L2, sample_frequency, f_lower, T_total)
    curves.append(dict(L1=L1, L2=L2, t=t_rel, phi=phi, amp=amp, peak=peak_idx))

def _slice_window(t_rel, y, w_sec):
    mask = (t_rel >= -w_sec) & (t_rel <= 0.0)
    return t_rel[mask], y[mask]

fig, ax = plt.subplots(figsize=(7, 4))

for item in curves:
    t_w, phi_w = _slice_window(item["t"], item["phi"], window_sec)
    label = r"$\Lambda_1={:d},\ \Lambda_2={:d}\ (\Lambda_\mathrm{{sum}}={:d})$".format(
        item["L1"], item["L2"], item["L1"] + item["L2"]
    )
    ax.plot(t_w, phi_w, lw=1.8, label=label)

ax.set_xlabel("Time (s)")
ax.set_ylabel("Cumulative phase Φ(t)")
sns.despine(ax=ax, top=True, right=True, left=False, bottom=False)
ax.legend(loc="best", fontsize=9, ncol=1)

ZOOM_X1, ZOOM_X2 = -0.005, 0
ZOOM_Y1, ZOOM_Y2 = 1980, 2018

rect = Rectangle((ZOOM_X1, ZOOM_Y1), ZOOM_X2 - ZOOM_X1, ZOOM_Y2 - ZOOM_Y1,
                 fill=False, ec="black", lw=1.4)
ax.add_patch(rect)

axins = inset_axes(ax,
                   width="35%", height="35%",  
                   loc="lower right",         
                   borderpad=0.8)              


for item in curves:
    t_w, phi_w = _slice_window(item["t"], item["phi"], window_sec)
    axins.plot(t_w, phi_w, lw=1.5)

axins.set_xlim(ZOOM_X1, ZOOM_X2)
axins.set_ylim(ZOOM_Y1, ZOOM_Y2)
axins.xaxis.set_major_formatter(FormatStrFormatter('%.3f'))
axins.yaxis.set_major_formatter(FormatStrFormatter('%.0f'))
axins.tick_params(labelsize=11, which="both", direction="in")
sns.despine(ax=axins, top=False, right=False, left=False, bottom=False)

con1 = ConnectionPatch(xyA=(ZOOM_X1, ZOOM_Y2), coordsA=ax.transData,
                       xyB=(0, 1), coordsB=axins.transAxes,
                       color="black", lw=1.0)
con2 = ConnectionPatch(xyA=(ZOOM_X2, ZOOM_Y2), coordsA=ax.transData,
                       xyB=(1, 1), coordsB=axins.transAxes,
                       color="black", lw=1.0)
ax.add_artist(con1); ax.add_artist(con2)

plt.tight_layout()
plt.show()




# delta_phi_time

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from Generate_waveform import predict_waveform_from_sample  

m1 = 1.40
m2 = 1.40
spin1 = {"x": 0.0, "y": 0.0, "z": 0.1}
spin2 = {"x": 0.0, "y": 0.0, "z": 0.1}
lambda_list = [0, 100, 200, 300, 400, 500]  

fs   = 4096.0      
f_low = 20.0      

out_png = "delta_phi_time.png"
def gw_phase_from_hp_hc(hp, hc):
    phi = np.unwrap(np.arctan2(hc, hp))
    return phi

def resample_to_length(y, new_len):
    n = len(y)
    if n == new_len:
        return y
    x_old = np.linspace(0.0, 1.0, n)
    x_new = np.linspace(0.0, 1.0, new_len)
    return np.interp(x_new, x_old, y)

def align_and_delta_phi(phi_cae, phi_ref):
    dphi = phi_cae - phi_ref
    dphi = dphi - dphi[0]
    return dphi

def cal_generate_BNS_waveform(approximant='IMRPhenomXP_NRTidalv2', **kwargs):
    waveform_args = {"approximant": approximant}
    for key, value in kwargs.items():
        if value is not None: 
            waveform_args[key] = value
    hp, hc = waveform.get_td_waveform(**waveform_args)
    return hp, hc

all_time = None
curves = []  

for L in lambda_list:

    x_sample = [[m1, m2, L, L,
                 spin1["x"], spin1["y"], spin1["z"],
                 spin2["x"], spin2["y"], spin2["z"]]]
    hp_cae, hc_cae = predict_waveform_from_sample(x_sample)
    hp_cae = np.asarray(hp_cae).ravel()
    hc_cae = np.asarray(hc_cae).ravel()
    N_cae  = len(hp_cae)

    hp_ref, hc_ref = cal_generate_BNS_waveform(
        mass1=m1, mass2=m2,
        delta_t=1.0/fs, f_lower=f_low,
        spin1x=spin1["x"], spin1y=spin1["y"], spin1z=spin1["z"],
        spin2x=spin2["x"], spin2y=spin2["y"], spin2z=spin2["z"],
        lambda1=L, lambda2=L,
    )
    hp_ref = np.asarray(hp_ref).ravel()
    hc_ref = np.asarray(hc_ref).ravel()
    N_ref  = len(hp_ref)

    phi_cae = gw_phase_from_hp_hc(hp_cae, hc_cae)
    phi_ref = gw_phase_from_hp_hc(hp_ref, hc_ref)

    phi_ref_rs = resample_to_length(phi_ref, N_cae)

    T_ref = N_ref / fs
    t = np.linspace(0.0, T_ref, N_cae)

    dphi = align_and_delta_phi(phi_cae, phi_ref_rs)
    curves.append((L+L, t, dphi))

plt.figure(figsize=(7.2, 4.2))

curves.sort(key=lambda x: x[0]) 
for Lsum, t, dphi in curves:
    lbl = rf"$\Lambda_1{=}\Lambda_2{=}\frac{{\Lambda_\mathrm{{tot}}}}2$, $\Lambda_\mathrm{{tot}}={Lsum}$"
    plt.plot(t, dphi, lw=2.0, label=lbl)

plt.xlabel("Time (s)")
plt.ylabel(r"$\Delta \Phi(t)\ \mathrm{(rad)}$")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9, ncol=1, loc="best")
plt.tight_layout()
plt.savefig(out_png, dpi=220, bbox_inches="tight")
plt.show()

print(f"[Saved] {out_png}")


# mismatch_total_lambda

In [ ]:

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from Generate_waveform import predict_waveform_from_sample


sns.set_style("whitegrid")

def cal_overlap_freq(h1, h2, dt=1.0, psd="o4", fmin=None, fmax=None, eps=1e-30):
    def _build_psd_on_grid(fgrid, psd_spec, eps_):
        if psd_spec is None or (np.isscalar(psd_spec) and float(psd_spec) == 1.0):
            Sn = np.ones_like(fgrid, dtype=float)
        elif isinstance(psd_spec, str) and psd_spec.lower() in ("aligo_o4high","o4","o4high"):
            f_file, asd = np.loadtxt("data/aligo_O4high.txt", unpack=True, comments="#", dtype=float)
            Sn_file = asd**2
            Sn = np.interp(fgrid, f_file, Sn_file, left=np.inf, right=np.inf)
        elif callable(psd_spec):
            Sn = np.asarray(psd_spec(fgrid), dtype=float)
            if Sn.shape != fgrid.shape: Sn = np.broadcast_to(Sn, fgrid.shape).astype(float)
        elif isinstance(psd_spec, (tuple, list)) and len(psd_spec) == 2:
            f_psd, Sn_psd = np.asarray(psd_spec[0], float), np.asarray(psd_spec[1], float)
            Sn = np.interp(fgrid, f_psd, Sn_psd, left=np.inf, right=np.inf)
        else:
            Sn = np.asarray(psd_spec, dtype=float)
            if Sn.shape != fgrid.shape:
                raise ValueError(f"psd array shape {Sn.shape} != freq shape {fgrid.shape}")
        return np.maximum(Sn, eps_)
    h1, h2 = np.asarray(h1, float), np.asarray(h2, float)
    n1, n2 = len(h1), len(h2)
    if n1 != n2:
        nmin = min(n1, n2)
        if abs(n1 - n2) <= 5:
            h1, h2 = h1[:nmin], h2[:nmin]
        else:
            raise ValueError(f"Length mismatch >5: len(h1)={n1}, len(h2)={n2}")
    n = len(h1)
    if n < 4: raise ValueError("Signal too short.")
    H1 = np.fft.rfft(h1) * dt
    H2 = np.fft.rfft(h2) * dt
    f  = np.fft.rfftfreq(n, d=dt)
    H1, H2, f = H1[1:], H2[1:], f[1:] 
    df = f[1] - f[0] if f.size > 1 else 1.0/(n*dt)
    Sn = _build_psd_on_grid(f, psd, eps)
    m = np.isfinite(Sn)
    if fmin is not None: m &= (f >= float(fmin))
    if fmax is not None: m &= (f <= float(fmax))
    if not np.any(m): raise ValueError("Empty frequency band after masking.")
    f, H1, H2, Sn = f[m], H1[m], H2[m], Sn[m]
    weight = np.ones_like(f)
    if n % 2 == 0 and f.size > 0 and np.isclose(f[-1], 0.5/dt): weight[-1] = 0.5
    w = weight / Sn
    inner_12 = 4.0 * np.real(np.sum(H1 * np.conjugate(H2) * w)) * df
    inner_11 = 4.0 * np.real(np.sum(H1 * np.conjugate(H1) * w)) * df
    inner_22 = 4.0 * np.real(np.sum(H2 * np.conjugate(H2) * w)) * df
    denom = np.sqrt(max(inner_11, eps) * max(inner_22, eps))
    overlap  = float(inner_12 / denom)
    mismatch = 1.0 - overlap
    return mismatch, overlap

rng = np.random.default_rng(1234)
OUTPUT = "mismatch_total_lambda.png"

fs    = 2048.0          
dt    = 1.0/fs
f_low = 50.0           
T     = 8.0            

Lambda_sum_grid = np.linspace(0, 500, 25)  
N_per = 60                                

m_min, m_max = 1.1, 1.9
chi_sigma = 0.15   
chi_clip  = 0.5

def _sample_mass_pair(rng):
    m1 = rng.uniform(m_min, m_max)
    m2 = rng.uniform(m_min, m_max)
    if m2 > m1: m1, m2 = m2, m1
    return m1, m2

def _sample_spin_vec(rng, mu_z=0.10):
    sx = np.clip(rng.normal(0.0, chi_sigma), -chi_clip, chi_clip)
    sy = np.clip(rng.normal(0.0, chi_sigma), -chi_clip, chi_clip)
    sz = np.clip(rng.normal(mu_z, chi_sigma), -chi_clip, chi_clip)
    return sx, sy, sz

def _split_lambda_sum(rng, Lsum):
    u = rng.uniform(0.25, 0.75)
    L1 = u * Lsum
    L2 = Lsum - L1
    return int(round(L1)), int(round(L2))

def _hp_numpy(hp):
    try:
        return np.asarray(hp, dtype=float)
    except Exception:
        return np.array(hp, dtype=float)

HAVE_RMC = False

cae_mean, cae_std = [], []
rmc_mean, rmc_std = [], []  

for Lsum in tqdm(Lambda_sum_grid, desc="Scanning Λ1+Λ2"):
    cae_mis = []
    rmc_mis = []

    for _ in range(N_per):
        m1, m2 = _sample_mass_pair(rng)
        s1x, s1y, s1z = _sample_spin_vec(rng)
        s2x, s2y, s2z = _sample_spin_vec(rng)
        L1, L2       = _split_lambda_sum(rng, Lsum)

        hp_ref, hc_ref = cal_generate_BNS_waveform(
            mass1=m1, mass2=m2,
            delta_t=dt, f_lower=f_low,
            spin1x=s1x, spin1y=s1y, spin1z=s1z,
            spin2x=s2x, spin2y=s2y, spin2z=s2z,
            lambda1=L1, lambda2=L2,
        )
        h_ref = _hp_numpy(hp_ref)

        x_sample = [[m1, m2, L1, L2, s1x, s1y, s1z, s2x, s2y, s2z]]
        hp_cae, hc_cae = predict_waveform_from_sample(x_sample)
        h_cae = _hp_numpy(hp_cae)

        try:
            mis_cae, _ = cal_overlap_freq(h_cae, h_ref, dt=dt, psd="o4", fmin=f_low, fmax=None)
            cae_mis.append(mis_cae)
        except Exception:
            continue
        if HAVE_RMC:
            hp_rmc, hc_rmc = predict_waveform_from_sample_rmc(x_sample)
            h_rmc = _hp_numpy(hp_rmc)
            try:
                mis_rmc, _ = cal_overlap_freq(h_rmc, h_ref, dt=dt, psd="o4", fmin=f_low, fmax=None)
                rmc_mis.append(mis_rmc)
            except Exception:
                pass

    cae_mis = np.array(cae_mis, dtype=float)
    cae_mean.append(np.mean(cae_mis) if cae_mis.size else np.nan)
    cae_std.append (np.std (cae_mis) if cae_mis.size else np.nan)

    if HAVE_RMC:
        rmc_mis = np.array(rmc_mis, dtype=float)
        rmc_mean.append(np.mean(rmc_mis) if rmc_mis.size else np.nan)
        rmc_std.append (np.std (rmc_mis) if rmc_mis.size else np.nan)

cae_mean = np.array(cae_mean); cae_std = np.array(cae_std)
if HAVE_RMC:
    rmc_mean = np.array(rmc_mean); rmc_std = np.array(rmc_std)

plt.figure(figsize=(7, 4))

plt.fill_between(Lambda_sum_grid, cae_mean - cae_std, cae_mean + cae_std, alpha=0.18, label="cAE (±1σ)")
plt.plot(Lambda_sum_grid, cae_mean, lw=2.0, label="cAE (mean)")

if HAVE_RMC:
    plt.fill_between(Lambda_sum_grid, rmc_mean - rmc_std, rmc_mean + rmc_std, alpha=0.18, label="Residual-MLP-CNN (±1σ)")
    plt.plot(Lambda_sum_grid, rmc_mean, lw=2.0, label="Residual-MLP-CNN (mean)")

plt.xlabel(r"$\Lambda_1+\Lambda_2$")
plt.ylabel("Mismatch (vs. IMRPhenomXP_NRTidalv2)")
plt.yscale("log")
sns.despine(top=True, right=True, left=False, bottom=False)
plt.grid(True, which="both", alpha=0.25)
plt.legend(loc="upper left", fontsize=9, ncol=1)
plt.tight_layout()
plt.savefig(OUTPUT, dpi=220, bbox_inches="tight")
plt.show()

print(f"[Done] Saved figure to {OUTPUT}")


# Bayesian inference efficiency

In [ ]:

import numpy as np
import emcee
import corner
import matplotlib.pyplot as plt
from pathlib import Path

from Generate_waveform import predict_waveform_from_sample


PSD_PATH   = "data/aligo_O4high.txt"  
FS         = 2048                   
FMIN, FMAX = 20.0, None               
N_WALKERS  = 48
N_BURN     = 1000
N_STEPS    = 4000
THIN       = 5
OUT_NPZ    = "posterior_cae_masses_o4.npz"  
T_OBS=2
m1_true, m2_true = 1.45, 1.25
lam1_true, lam2_true = 200.0, 200.0
s1x, s1y, s1z = 0.0, 0.0, 0.1
s2x, s2y, s2z = 0.0, 0.0, 0.1

def _pad_or_tail(x, target_len):
    x = np.asarray(x, dtype=np.float64)
    if x.size >= target_len:
        return x[-target_len:]
    out = np.zeros(target_len, dtype=np.float64)
    out[-x.size:] = x
    return out

def _align_to_merger(hp, hc, fs, T_window):
    hp = np.asarray(hp, dtype=np.float64)
    hc = np.asarray(hc, dtype=np.float64)
    amp = np.hypot(hp, hc)
    peak = int(np.argmax(amp))
    t = np.arange(hp.size, dtype=np.float64) / fs
    t_rel = t - t[peak]
    m = (t_rel >= -T_window) & (t_rel <= 0.0)
    return t_rel[m], hp[m], hc[m]

def load_psd_asd_square(psd_path):
    f, asd = np.loadtxt(psd_path, unpack=True)
    return f.astype(float), (asd.astype(float))**2

def build_Sn_grid(n, dt, psd_path, fmin=None, fmax=None, eps=1e-30):
    f_psd, Sn_psd = load_psd_asd_square(psd_path)
    freqs = np.fft.rfftfreq(n, d=dt) 
    freqs = freqs[1:]                 
    Sn = np.interp(freqs, f_psd, Sn_psd, left=np.inf, right=np.inf)
    Sn = np.maximum(Sn, eps)
    mask = np.isfinite(Sn)
    if fmin is not None: mask &= (freqs >= float(fmin))
    if fmax is not None: mask &= (freqs <= float(fmax))
    return freqs[mask], Sn[mask], mask

def inner_product(a, b, dt, psd_path, fmin=None, fmax=None):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    n = min(a.size, b.size)
    a = a[:n]; b = b[:n]
    A = np.fft.rfft(a) * dt
    B = np.fft.rfft(b) * dt
    freqs_all = np.fft.rfftfreq(n, d=dt)
    A = A[1:]; B = B[1:]; freqs_all = freqs_all[1:]
    fgrid, Sn, mask = build_Sn_grid(n, dt, psd_path, fmin, fmax)
    A = A[mask]; B = B[mask]
    if A.size == 0:
        raise ValueError("Empty band after PSD masking.")
    df = fgrid[1] - fgrid[0] if fgrid.size > 1 else 1.0/(n*dt)
    weight = np.ones_like(fgrid)
    if n % 2 == 0 and fgrid.size > 0 and np.isclose(fgrid[-1], 0.5/dt):
        weight[-1] = 0.5
    w = weight / Sn
    return 4.0 * np.real(np.sum(A * np.conjugate(B) * w)) * df

def make_strain_from_hphc(hp, hc, Fp=1.0, Fx=0.0):
    return Fp*np.asarray(hp, float) + Fx*np.asarray(hc, float)

def gen_waveform_cae(m1, m2, L1, L2, s1x, s1y, s1z, s2x, s2y, s2z):
    x = [[float(m1), float(m2), float(L1), float(L2),
          float(s1x), float(s1y), float(s1z),
          float(s2x), float(s2y), float(s2z)]]
    hp, hc = predict_waveform_from_sample(x)
    return np.asarray(hp, float), np.asarray(hc, float)

dt = 1.0 / FS
hp_true, hc_true = gen_waveform_cae(m1_true, m2_true, lam1_true, lam2_true, s1x, s1y, s1z, s2x, s2y, s2z)
t_rel, hp_win, hc_win = _align_to_merger(hp_true, hc_true, FS, T_OBS)
d = make_strain_from_hphc(hp_win, hc_win, Fp=1.0, Fx=0.0)  

TARGET_LEN = d.size

def waveform_windowed(m1, m2):
    hp, hc = gen_waveform_cae(m1, m2, lam1_true, lam2_true, s1x, s1y, s1z, s2x, s2y, s2z)
    _, hpw, hcw = _align_to_merger(hp, hc, FS, T_OBS)
    h = make_strain_from_hphc(hpw, hcw, 1.0, 0.0)
    return _pad_or_tail(h, TARGET_LEN)

m_min, m_max = 1.0, 3.0

def log_prior(theta):
    m1, m2 = theta
    if not (m_min <= m1 <= m_max and m_min <= m2 <= m_max):
        return -np.inf
    if m1 < m2:  
        return -np.inf
    return 0.0

def log_likelihood(theta):
    m1, m2 = theta
    h = waveform_windowed(m1, m2)
    resid = d - h
    return -0.5 * inner_product(resid, resid, dt, PSD_PATH, fmin=FMIN, fmax=FMAX)

def log_posterior(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta)

rng = np.random.default_rng(20250909)
pos = np.vstack([
    rng.normal(m1_true, 0.05, size=N_WALKERS),
    rng.normal(m2_true, 0.05, size=N_WALKERS),
]).T
for i in range(N_WALKERS):
    m1, m2 = np.clip(pos[i], m_min, m_max)
    if m1 < m2: m1, m2 = m2, m1
    pos[i] = (m1, m2)

sampler = emcee.EnsembleSampler(N_WALKERS, 2, log_posterior, moves=[emcee.moves.StretchMove(a=2.0)])
pos, _, _ = sampler.run_mcmc(pos, N_BURN, progress=True)
sampler.reset()
sampler.run_mcmc(pos, N_STEPS, progress=True)

samples = sampler.get_chain(flat=True, thin=THIN)

np.savez(
    OUT_NPZ,
    samples=samples,
    labels=np.array([r"$m_1\,[M_\odot]$", r"$m_2\,[M_\odot]$"], dtype=object),
    truths=np.array([m1_true, m2_true], dtype=float),
    config=np.array([FS, T_OBS, FMIN if FMIN is not None else -1.0,
                     FMAX if FMAX is not None else -1.0], dtype=float)
)
print(f"[saved] posterior -> {OUT_NPZ}  (#samples={samples.shape[0]})")



In [ ]:

import os
os.environ.setdefault("OMP_NUM_THREADS", "1")

import numpy as np
import emcee
import corner
import matplotlib.pyplot as plt
from pathlib import Path
import multiprocessing as mp

from pycbc import waveform 

PSD_PATH   = "data/aligo_O4high.txt"  
FS         = 2048                     
T_OBS      = 2.0                      
FMIN, FMAX = 20.0, None               
N_WALKERS  = 48
N_BURN     = 2000
N_STEPS    = 4000
THIN       = 5
N_PROCS    = 24                        
OUT_NPZ    = "posterior_traditional_masses_o4.npz"

m1_true, m2_true = 1.45, 1.25
lam1_true, lam2_true = 200.0, 200.0
s1x, s1y, s1z = 0.0, 0.0, 0.1
s2x, s2y, s2z = 0.0, 0.0, 0.1
distance = 100.0       
inclination = 0.0      
coa_phase   = 0.0     

def cal_generate_BNS_waveform(approximant='IMRPhenomXP_NRTidalv2', **kwargs):
    waveform_args = {"approximant": approximant}
    for k, v in kwargs.items():
        if v is not None:
            waveform_args[k] = v
    hp, hc = waveform.get_td_waveform(**waveform_args)
    return hp, hc

def gen_waveform_traditional(m1, m2, L1, L2, s1x, s1y, s1z, s2x, s2y, s2z):
    hp, hc = cal_generate_BNS_waveform(
        approximant='IMRPhenomXP_NRTidalv2',
        mass1=float(m1), mass2=float(m2),
        delta_t=1.0/FS, f_lower=FMIN if FMIN is not None else 20.0,
        spin1x=float(s1x), spin1y=float(s1y), spin1z=float(s1z),
        spin2x=float(s2x), spin2y=float(s2y), spin2z=float(s2z),
        lambda1=float(L1), lambda2=float(L2),
        distance=float(distance), inclination=float(inclination), coa_phase=float(coa_phase),
    )
    return np.asarray(hp, float), np.asarray(hc, float)

def _pad_or_tail(x, target_len):
    x = np.asarray(x, dtype=np.float64)
    if x.size >= target_len:
        return x[-target_len:]
    out = np.zeros(target_len, dtype=np.float64)
    out[-x.size:] = x
    return out

def _align_to_merger(hp, hc, fs, T_window):
    hp = np.asarray(hp, dtype=np.float64)
    hc = np.asarray(hc, dtype=np.float64)
    amp = np.hypot(hp, hc)
    peak = int(np.argmax(amp))
    t = np.arange(hp.size, dtype=np.float64) / fs
    t_rel = t - t[peak]
    m = (t_rel >= -T_window) & (t_rel <= 0.0)
    return t_rel[m], hp[m], hc[m]

def load_psd_asd_square(psd_path):
    f, asd = np.loadtxt(psd_path, unpack=True)
    return f.astype(float), (asd.astype(float))**2

def build_Sn_grid(n, dt, psd_path, fmin=None, fmax=None, eps=1e-30):
    f_psd, Sn_psd = load_psd_asd_square(psd_path)
    freqs = np.fft.rfftfreq(n, d=dt)[1:] 
    Sn = np.interp(freqs, f_psd, Sn_psd, left=np.inf, right=np.inf)
    Sn = np.maximum(Sn, eps)
    mask = np.isfinite(Sn)
    if fmin is not None: mask &= (freqs >= float(fmin))
    if fmax is not None: mask &= (freqs <= float(fmax))
    return freqs[mask], Sn[mask], mask

def inner_product(a, b, dt, psd_path, fmin=None, fmax=None):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    n = min(a.size, b.size)
    a = a[:n]; b = b[:n]
    A = (np.fft.rfft(a) * dt)[1:] 
    B = (np.fft.rfft(b) * dt)[1:]
    fgrid, Sn, mask = build_Sn_grid(n, dt, psd_path, fmin, fmax)
    A = A[mask]; B = B[mask]
    if A.size == 0:
        raise ValueError("Empty band after PSD masking.")
    df = fgrid[1] - fgrid[0] if fgrid.size > 1 else 1.0/(n*dt)
    weight = np.ones_like(fgrid)
    if n % 2 == 0 and fgrid.size > 0 and np.isclose(fgrid[-1], 0.5/dt):
        weight[-1] = 0.5
    w = weight / Sn
    return 4.0 * np.real(np.sum(A * np.conjugate(B) * w)) * df

def make_strain_from_hphc(hp, hc, Fp=1.0, Fx=0.0):
    return Fp*np.asarray(hp, float) + Fx*np.asarray(hc, float)

dt = 1.0 / FS
hp_true, hc_true = gen_waveform_traditional(m1_true, m2_true, lam1_true, lam2_true, s1x, s1y, s1z, s2x, s2y, s2z)
t_rel, hp_win, hc_win = _align_to_merger(hp_true, hc_true, FS, T_OBS)
d = make_strain_from_hphc(hp_win, hc_win, Fp=1.0, Fx=0.0)
TARGET_LEN = d.size

def waveform_windowed(m1, m2):
    hp, hc = gen_waveform_traditional(m1, m2, lam1_true, lam2_true, s1x, s1y, s1z, s2x, s2y, s2z)
    _, hpw, hcw = _align_to_merger(hp, hc, FS, T_OBS)
    h = make_strain_from_hphc(hpw, hcw, 1.0, 0.0)
    return _pad_or_tail(h, TARGET_LEN)

m_min, m_max = 1.0, 3.0

def log_prior(theta):
    m1, m2 = theta
    if not (m_min <= m1 <= m_max and m_min <= m2 <= m_max):
        return -np.inf
    if m1 < m2:
        return -np.inf
    return 0.0

def log_likelihood(theta):
    m1, m2 = theta
    h = waveform_windowed(m1, m2)
    resid = d - h
    return -0.5 * inner_product(resid, resid, dt, PSD_PATH, fmin=FMIN, fmax=FMAX)

def log_posterior(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta)

def run_sampler():
    rng = np.random.default_rng(20250910)
    pos = np.vstack([
        rng.normal(m1_true, 0.05, size=N_WALKERS),
        rng.normal(m2_true, 0.05, size=N_WALKERS),
    ]).T
    for i in range(N_WALKERS):
        m1, m2 = np.clip(pos[i], m_min, m_max)
        if m1 < m2: m1, m2 = m2, m1
        pos[i] = (m1, m2)

    with mp.get_context("spawn").Pool(processes=N_PROCS) as pool:
        sampler = emcee.EnsembleSampler(N_WALKERS, 2, log_posterior,
                                        moves=[emcee.moves.StretchMove(a=2.0)], pool=pool)
        pos, _, _ = sampler.run_mcmc(pos, N_BURN, progress=True)
        sampler.reset()
        sampler.run_mcmc(pos, N_STEPS, progress=True)
        samples = sampler.get_chain(flat=True, thin=THIN)
    return samples

samples = run_sampler()
np.savez(
    OUT_NPZ,
    samples=samples,
    labels=np.array([r"$m_1\,[M_\odot]$", r"$m_2\,[M_\odot]$"], dtype=object),
    truths=np.array([m1_true, m2_true], dtype=float),
    config=np.array([FS, T_OBS, FMIN if FMIN is not None else -1.0,
                        FMAX if FMAX is not None else -1.0], dtype=float)
)
print(f"[saved] posterior -> {OUT_NPZ}  (#samples={samples.shape[0]})")




In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import corner

CAE_NPZ = "posterior_cae_masses_o4.npz"
TRD_NPZ = "posterior_traditional_masses_o4.npz"

cae = np.load(CAE_NPZ, allow_pickle=True)
trd = np.load(TRD_NPZ, allow_pickle=True)

chain_dl = cae["samples"]             
chain_tr = trd["samples"]             

labels = cae["labels"].tolist() if "labels" in cae else [r"$m_1\,[M_\odot]$", r"$m_2\,[M_\odot]$"]
truths = cae["truths"].tolist() if "truths" in cae else [1.45, 1.25]

def union_range(a, b, pad=0.05):
    lo = np.minimum(a.min(axis=0), b.min(axis=0))
    hi = np.maximum(a.max(axis=0), b.max(axis=0))
    wid = hi - lo
    lo = lo - pad * wid
    hi = hi + pad * wid
    return [(float(lo[i]), float(hi[i])) for i in range(a.shape[1])]

my_range = union_range(chain_dl, chain_tr, pad=0.05)

figure = corner.corner(
    chain_dl, labels=labels,
    bins=50,
    smooth=0.7,
    range=my_range,
    truth_color='black',
    truths=truths,
    show_titles=True,
    title_kwargs={"fontsize": 12},
    color="#1D90FDFF",  
    verbose=False
)

corner.corner(
    chain_tr, labels=labels,
    bins=50,
    smooth=0.7,
    range=my_range,
    truth_color='black',
    truths=truths,
    show_titles=True,
    title_kwargs={"fontsize":12},
    color="#696969FF",  
    verbose=False,
    fig=figure        
)

legend_items = [
    Patch(color="#1D90FDFF", label="cAE"),
    Patch(color="#696969FF", label="IMRPhenomXP_NRTidalv2"),
]
plt.legend(handles=legend_items, loc="upper right",
           bbox_to_anchor=(1.5, 2.0), prop={'size': 12})

plt.show()
